# 02 - Analisis Exploratorio (EDA)

Dataset: siniestros viales CABA 2019-2025, enriquecido con victimas y clima.

Preguntas:
1. Como se distribuyen los siniestros en el tiempo?
2. Que vias, franjas horarias y tipos de colision son mas letales?
3. Que perfil de victima se asocia a gravedad fatal?
4. El clima influye sobre la gravedad?
5. Donde se concentran los hotspots?

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'siniestros_features.parquet')
print(f'{len(df):,} siniestros | {df["target"].sum():,} fatales ({df["target"].mean():.2%})')
print(f'Periodo: {df["fecha"].min():%Y-%m} a {df["fecha"].max():%Y-%m}')

## 1. Distribucion temporal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df['año'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Siniestros por año')
axes[0].bar_label(axes[0].containers[0], fontsize=8)
df['mes'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Por mes')
df['dia_semana'].value_counts().sort_index().plot(kind='bar', ax=axes[2], title='Por dia (0=lun)')
plt.tight_layout(); plt.show()

In [ ]:
# Siniestros y tasa de fatalidad por hora del dia
por_hora = df.groupby('hora_num')['target'].agg(['size', 'mean'])
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(por_hora.index, por_hora['size'], alpha=0.5, label='Cantidad')
ax1.set_xlabel('Hora'); ax1.set_ylabel('Siniestros')
ax2 = ax1.twinx()
ax2.plot(por_hora.index, por_hora['mean']*100, 'r-o', ms=3, label='% fatal')
ax2.set_ylabel('% fatal')
plt.title('Volumen y letalidad por hora del dia')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.show()

**Hallazgo:** el volumen pico es horario laboral, pero la **letalidad pica en madrugada** (~2.5% vs ~0.8% tarde): menos siniestros pero mas graves por exceso de velocidad y menor transito.

## 2. Que hace mas letal a un siniestro?

In [ ]:
def tasa_por(col, min_n=200):
    t = df.groupby(col)['target'].agg(['mean', 'size'])
    t = t[t['size'] >= min_n]
    return (t['mean']*100).round(2).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
tasa_por('tipo_calle').plot(kind='barh', ax=axes[0], title='% fatal por tipo de via')
tasa_por('franja_horaria').plot(kind='barh', ax=axes[1], title='% fatal por franja horaria')
tasa_por('participantes', min_n=500).head(10).plot(kind='barh', ax=axes[2], title='% fatal por participacion (top 10)')
plt.tight_layout(); plt.show()

print('Autopista vs Calle:', round(tasa_por('tipo_calle')['AUTOPISTA'] / tasa_por('tipo_calle')['CALLE'], 1), 'x')
print('Madrugada vs Tarde:', round(tasa_por('franja_horaria')['madrugada'] / tasa_por('franja_horaria')['tarde'], 1), 'x')

In [ ]:
# Colisiones mas peligrosas: combinaciones vehiculo-victima
top = tasa_por('participantes', min_n=300)
display(top.head(12))

**Hallazgos:**
- Autopistas ~6.7% letal vs calles ~0.8% (**8.6x**) - velocidad de impacto.
- Madrugada **3.3x** mas letal que la tarde.
- Las peores colisiones involucran **peaton-camion** (9.1%), **moto contra objeto fijo** (8.1%) y **peaton-transporte publico** (8.0%).

## 3. Perfil de la victima: fatal vs no fatal

In [ ]:
perfil = df.groupby('target')[['pct_moto', 'pct_peaton', 'edad_promedio',
                               'pct_sexo_m', 'n_victimas']].mean().T.round(3)
perfil.columns = ['no fatal (0)', 'fatal (1)']
display(perfil)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, titulo in [
    (axes[0], 'pct_peaton', '% peatones'),
    (axes[1], 'pct_moto', '% motos'),
    (axes[2], 'edad_promedio', 'edad promedio'),
]:
    sns.kdeplot(data=df, x=col, hue='target', common_norm=False, fill=True, ax=ax)
    ax.set_title(titulo); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

**Hallazgos:** en siniestros fatales, las victimas son mucho mas seguido **peatones** (39% vs 12%), hay mayor proporcion de **motos**, la edad media sube (45 vs 38 anos) y predominan varones (75% vs 51%). Este perfil explica el poder predictivo de estas variables en el modelo.

## 4. Clima y gravedad

In [ ]:
print(df[['lluvia_mm', 'temp_max']].describe().round(1))
for col in ['lluvia_mm', 'temp_max']:
    print(f'correlacion {col}-target:', round(df[[col, 'target']].corr().iloc[0, 1], 4))

# Letalidad segun lluvia del dia
df['lluvia_bin'] = pd.cut(df['lluvia_mm'], bins=[-0.1, 0, 5, 20, 100],
                          labels=['seco', 'llovizna (0-5mm)', 'lluvia (5-20mm)', 'tormenta (>20mm)'])
tasa_lluvia = df.groupby('lluvia_bin', observed=True)['target'].agg(['mean', 'size'])
tasa_lluvia['mean_pct'] = (tasa_lluvia['mean']*100).round(2)
display(tasa_lluvia)

**Hallazgo (negativo, importante):** la correlacion del clima con la gravedad es practicamente nula (<0.005). El clima podria afectar la *cantidad* de siniestros, no tanto su *severidad*. Documentar hallazgos negativos tambien aporta evidencia.

## 5. Hotspots geoespaciales

In [ ]:
# Ranking por comuna (con minimo de casos para robustez)
c = df.groupby('comuna')['target'].agg(['mean', 'size'])
c = c[c['size'] >= 500].sort_values('mean', ascending=False)
c['mean_pct'] = (c['mean']*100).round(2)
display(c[['mean_pct', 'size']].head(15))

fig, ax = plt.subplots(figsize=(10, 4))
c['mean_pct'].plot(kind='bar', ax=ax, title='% fatal por comuna')
ax.bar_label(ax.containers[0], fmt='%.1f', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Mapa de densidad: fatales (rojo) vs resto (azul)
import folium
from folium.plugins import HeatMap

geo = df.dropna(subset=['latitud', 'longitud'])
center = [geo['latitud'].mean(), geo['longitud'].mean()]

m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')
HeatMap(geo[geo.target == 0][['latitud', 'longitud']].values,
        radius=7, blur=10, name='No fatales', gradient={0.4: 'blue', 0.65: 'cyan'}).add_to(m)
HeatMap(geo[geo.target == 1][['latitud', 'longitud']].values,
        radius=9, blur=12, name='Fatales', gradient={0.4: 'orange', 0.65: 'red'}).add_to(m)
folium.LayerControl().add_to(m)
m.save(PROJECT_ROOT / 'reports' / 'figures' / 'mapa_hotspots.html')
m

**Hallazgos:** Comuna 8 (Villa Soldati, zona sur) lidera en letalidad (2.1%), seguida por 4 y 10 - correspondencia con avenidas de alta velocidad y menor infraestructura peatonal. El mapa interactivo queda guardado en `reports/figures/mapa_hotspots.html`.

## Conclusiones del EDA

1. **Letalidad no es volumen**: madrugada y autopistas concentran pocos siniestros pero muy graves.
2. **El perfil de victima predice**: peaton/moto + edad avanzada = mayor riesgo fatal. Es la senal mas fuerte del dataset.
3. **Colisiones criticas identificables**: peaton-camion y moto-objeto fijo superan 8% de letalidad -> objetivos directos de politica publica (separacion de trafico pesado, proteccion motociclista).
4. **Clima irrelevante para severidad**: hallazgo negativo documentado.
5. **Geografia desigual**: zona sur de la ciudad (comunas 8, 4, 10) con tasas hasta 2.7x mayores que el norte.